In [1]:
from myutils import *
from scipy.stats import ortho_group
from scipy.optimize import minimize
from tqdm import tqdm

from scipy.integrate import quad

In [2]:
class Psi:
    def __init__(self, x, mu, sigma):
        self.x = x
        self.mu = mu
        self.sigma = sigma

        psi_vec = Psi.func(x, mu, sigma)
        self.psi_vec = psi_vec / np.linalg.norm(psi_vec)

        psi_prime_vec = Psi.prime_func(x, mu, sigma)
        self.psi_prime_vec = psi_prime_vec / np.linalg.norm(psi_prime_vec)

        self.basis = np.column_stack([psi_vec, psi_prime_vec])
        self.basis_proj = self.basis @ self.basis.conj().T


    @staticmethod
    def func(x, mu, sigma):
        return np.exp(-(x-mu)**2 / (4*sigma**2)) / (2*np.pi*sigma**2)**.25


    @staticmethod
    def prime_func(x, mu, sigma):
        prefactor = 1 / (2 * (2 * np.pi)**.25)
        exp_part = np.exp(- (x - mu)**2 / (4*sigma**2))
        power_part = (1 / sigma**2)**(5 / 4)
        return prefactor * exp_part * (x - mu) * power_part



In [ ]:
class POVM:
    def __init__(self, psi: Psi):
        U = ortho_group.rvs(dim=2)

        mat = psi.basis @ U
        self.E1, self.E2 = np.outer(mat[:, 0].T, mat[:, 0]), np.outer(mat[:, 1].T, mat[:, 1])

        if not np.allclose(self.E1 + self.E2, psi.basis_proj):
            raise RuntimeError

In [ ]:
class Measure:
    def __init__(self, psi: Psi, povm: POVM):
        self.psi = psi
        self.povm = povm

        self.p1 = np.real(psi.psi_vec.conj().T @ povm.E1 @ psi.psi_vec)
        self.p2 = np.real(psi.psi_vec.conj().T @ povm.E2 @ psi.psi_vec)


    def gen(self, photons: int, noise: int, repeat: int):
        self.photons = photons
        self.noise = noise
        self.repeat = repeat
        outcomes = []
        for _ in range(repeat):
            data = np.histogram(np.random.uniform(0, 1, np.random.poisson(photons)), [0, self.p1, self.p1 + self.p2])[0] + \
                   np.random.poisson(noise, 2)
            outcomes.append(data)
        self.outcomes = np.array(outcomes)
        return self.outcomes


    def est(self, retry=5):
        def mle(data):
            def nll(theta):
                psi = Psi(self.psi.x, theta, self.psi.sigma)
                m = Measure(psi, self.povm)

                Lam1 = m.p1 * self.photons + self.noise
                Lam2 = m.p2 * self.photons + self.noise
                Lam = np.array([Lam1, Lam2])

                return - data @ np.log(Lam).T + np.sum(Lam)

            for _ in range(retry):
                opt = minimize(nll, x0=np.random.uniform(self.psi.x.min(), self.psi.x.max(), 1))
                if opt.success:
                    break

            if opt.success:
                return opt.x.item()
            else:
                return np.nan

        result = [] 
        for i in range(self.repeat):
            result.append(mle(self.outcomes[i]))
        self.result = np.array(result)
        return self.result



In [15]:
psi = Psi(np.arange(-50, 50, 1), 0, 10)

sim = []
for _ in tqdm(range(10)):
    m = Measure(psi, POVM(psi))
    m.gen(100, 10, 100)
    m.est()

    sim.append(np.nanvar(m.est(), ddof=1) * 100 / 10**2)

100%|██████████| 10/10 [00:17<00:00,  1.79s/it]


In [14]:
np.mean(sim)

19.08128056522219

In [16]:
np.mean(sim)

21.20194513319615

In [17]:
21.2 / 19

1.1157894736842104